In [ ]:
# OpenAI Careers Job Listings - Ashby

## 1. Project Overview

This project uses Python to automatically collect publicly available job listing information from a company's careers website. The scraped data will be transformed into a structured dataset, cleaned, analyzed, and visualized to identify patterns in hiring demand, job locations, and advertised compensation.


In [51]:
# first import
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [52]:
# Retrieve the data

url = "https://api.ashbyhq.com/posting-api/job-board/openai?includeCompensation=true"
response = requests.get(url)
response.status_code

200

In [55]:
# response

data = response.json()
type(data)

dict

In [56]:
data.keys()

dict_keys(['jobs', 'apiVersion'])

In [57]:
# jobs retrieved

len(data["jobs"])


807

In [58]:
data["jobs"][0]

{'id': '8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3',
 'title': 'Technical Program Manager, Compute Infrastructure',
 'department': 'Technical Program Management',
 'team': 'Technical Program Management',
 'employmentType': 'FullTime',
 'location': 'San Francisco',
 'shouldDisplayCompensationOnJobPostings': True,
 'secondaryLocations': [],
 'publishedAt': '2026-03-12T16:38:15.322+00:00',
 'isListed': True,
 'isRemote': None,
 'workplaceType': None,
 'address': {'postalAddress': {'addressRegion': 'California',
   'addressCountry': 'United States',
   'addressLocality': 'San Francisco'}},
 'jobUrl': 'https://jobs.ashbyhq.com/openai/8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3',
 'applyUrl': 'https://jobs.ashbyhq.com/openai/8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3/application',
 'descriptionHtml': '<h3><strong>About the Team</strong></h3><p style="min-height:1.5em">The compute infrastructure team runs the GPU fleet and large-scale compute clusters that serve the models backing ChatGPT and the API, while als

In [59]:
# dataset structure

jobs = data["jobs"]

records = []

for job in jobs:
    compensation = job.get("compensation") or {}
    salary_components = compensation.get("summaryComponents") or []

    salary = next(
        (
            component
            for component in salary_components
            if component.get("compensationType") == "Salary"
        ),
        {}
    )

    salary_min = salary.get("minValue")
    salary_max = salary.get("maxValue")

    records.append({
        "Job Title": job.get("title"),
        "Department": job.get("department"),
        "Team": job.get("team"),
        "Employment Type": job.get("employmentType"),
        "Location": job.get("location"),
        "Workplace Type": job.get("workplaceType"),
        "Published Date": job.get("publishedAt"),
        "Salary Min": salary_min,
        "Salary Max": salary_max,
        "Job URL": job.get("jobUrl")
    })

df = pd.DataFrame(records)

df.head()

,Job Title,Department,Team,Employment Type,Location,Workplace Type,Published Date,Salary Min,Salary Max,Job URL
0,"Technical Program Manager, Compute Infrastructure",Technical Program Management,Technical Program Management,FullTime,San Francisco,None,2026-03-12T16:38:15.322+00:00,257000.0,335000.0,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...
1,Research Engineer,Research,Research,FullTime,San Francisco,None,2025-04-05T00:03:20.653+00:00,250000.0,445000.0,https://jobs.ashbyhq.com/openai/240d459b-696d-...
2,Account Director - Tokyo,Go To Market,Sales,FullTime,"Tokyo, Japan",None,2026-01-23T00:17:18.483+00:00,NaN,NaN,https://jobs.ashbyhq.com/openai/18f58952-c242-...
3,"Research Engineer, Retrieval & Search, Applied...",Applied AI,Applied AI Engineering,FullTime,San Francisco,None,2024-03-20T21:33:20.763+00:00,293000.0,585000.0,https://jobs.ashbyhq.com/openai/7322d344-9325-...
4,"Account Director, Startups",Go To Market,Sales,FullTime,São Paulo,Hybrid,2026-08-31T21:22:23.258+00:00,NaN,NaN,https://jobs.ashbyhq.com/openai/0b428c6d-7c06-...


In [60]:
df.shape

(807, 10)

In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 807 entries, 0 to 806
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Job Title        807 non-null    object 
 1   Department       807 non-null    object 
 2   Team             807 non-null    object 
 3   Employment Type  807 non-null    object 
 4   Location         807 non-null    object 
 5   Workplace Type   561 non-null    object 
 6   Published Date   807 non-null    object 
 7   Salary Min       670 non-null    float64
 8   Salary Max       670 non-null    float64
 9   Job URL          807 non-null    object 
dtypes: float64(2), object(8)
memory usage: 63.2+ KB


In [62]:
df.isna().sum()

Job Title            0
Department           0
Team                 0
Employment Type      0
Location             0
Workplace Type     246
Published Date       0
Salary Min         137
Salary Max         137
Job URL              0
dtype: int64

In [63]:
# check for duplicates

df.duplicated().sum()
df["Job URL"].duplicated().sum()

0

In [64]:
# convert published date

df["Published Date"] = pd.to_datetime(df["Published Date"])
df["Published Date"].dtype

datetime64[ns, UTC]

In [65]:
# salary midpoint

df["Salary Midpoint"] = (
    df["Salary Min"] + df["Salary Max"]
) / 2
df[["Salary Min", "Salary Max", "Salary Midpoint"]].head(10)

,Salary Min,Salary Max,Salary Midpoint
0,257000.0,335000.0,296000.0
1,250000.0,445000.0,347500.0
2,NaN,NaN,NaN
3,293000.0,585000.0,439000.0
4,NaN,NaN,NaN
5,380000.0,500000.0,440000.0
6,266000.0,445000.0,355500.0
7,266000.0,445000.0,355500.0
8,295000.0,500000.0,397500.0
9,210000.0,490000.0,350000.0


In [66]:
# workplace type

df["Workplace Type"] = df["Workplace Type"].fillna("Not Specified")
df["Workplace Type"].value_counts()

Workplace Type
Hybrid           499
Not Specified    246
OnSite            33
Remote            29
Name: count, dtype: int64

In [67]:
df["Salary Midpoint"].isna().sum()

137

In [68]:
# clean data check

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 807 entries, 0 to 806
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   Job Title        807 non-null    object             
 1   Department       807 non-null    object             
 2   Team             807 non-null    object             
 3   Employment Type  807 non-null    object             
 4   Location         807 non-null    object             
 5   Workplace Type   807 non-null    object             
 6   Published Date   807 non-null    datetime64[ns, UTC]
 7   Salary Min       670 non-null    float64            
 8   Salary Max       670 non-null    float64            
 9   Job URL          807 non-null    object             
 10  Salary Midpoint  670 non-null    float64            
dtypes: datetime64[ns, UTC](1), float64(3), object(7)
memory usage: 69.5+ KB


,Job Title,Department,Team,Employment Type,Location,Workplace Type,Published Date,Salary Min,Salary Max,Job URL,Salary Midpoint
0,"Technical Program Manager, Compute Infrastructure",Technical Program Management,Technical Program Management,FullTime,San Francisco,Not Specified,2026-03-12 16:38:15.322000+00:00,257000.0,335000.0,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...,296000.0
1,Research Engineer,Research,Research,FullTime,San Francisco,Not Specified,2025-04-05 00:03:20.653000+00:00,250000.0,445000.0,https://jobs.ashbyhq.com/openai/240d459b-696d-...,347500.0
2,Account Director - Tokyo,Go To Market,Sales,FullTime,"Tokyo, Japan",Not Specified,2026-01-23 00:17:18.483000+00:00,NaN,NaN,https://jobs.ashbyhq.com/openai/18f58952-c242-...,NaN
3,"Research Engineer, Retrieval & Search, Applied...",Applied AI,Applied AI Engineering,FullTime,San Francisco,Not Specified,2024-03-20 21:33:20.763000+00:00,293000.0,585000.0,https://jobs.ashbyhq.com/openai/7322d344-9325-...,439000.0
4,"Account Director, Startups",Go To Market,Sales,FullTime,São Paulo,Hybrid,2026-08-31 21:22:23.258000+00:00,NaN,NaN,https://jobs.ashbyhq.com/openai/0b428c6d-7c06-...,NaN


In [69]:
df.describe(include="all")

,Job Title,Department,Team,Employment Type,Location,Workplace Type,Published Date,Salary Min,Salary Max,Job URL,Salary Midpoint
count,807,807,807,807,807,807,807,670.000000,670.000000,807,670.000000
unique,765,32,81,2,25,4,NaN,NaN,NaN,807,NaN
top,Applied AI Architect,Go To Market,Applied AI Engineering,FullTime,San Francisco,Hybrid,NaN,NaN,NaN,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...,NaN
freq,5,142,59,806,572,499,NaN,NaN,NaN,1,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-03 12:02:27.610586112+00:00,263270.269493,364486.962254,NaN,313878.615873
min,NaN,NaN,NaN,NaN,NaN,NaN,2023-05-25 04:06:29.356000+00:00,0.000000,70.000000,NaN,63.500000
25%,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-20 06:34:21.460000+00:00,216500.000000,285000.000000,NaN,249625.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-29 19:01:49.372999936+00:00,257000.000000,370000.000000,NaN,307500.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-26 17:45:50.788499968+00:00,295000.000000,445000.000000,NaN,360000.000000
max,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-15 11:25:38.610000+00:00,675000.000000,850000.000000,NaN,712500.000000


In [ ]:
df.to_csv("openai_job_listings.csv", index=False)

In [ ]:
# 2. Exploratory Analysis

In [ ]:
## jobs by department

department_counts = (
    df["Department"]
    .value_counts()
    .reset_index()
)

department_counts.columns = ["Department", "Job Count"]

department_counts

In [ ]:
department_counts.head(10)

In [70]:
## jobs by location

location_counts = (
    df["Location"]
    .value_counts()
    .reset_index()
)

location_counts.columns = ["Location", "Job Count"]

location_counts.head(15)

,Location,Job Count
0,San Francisco,572
1,Singapore,32
2,US - Remote,24
3,"Tokyo, Japan",23
4,"Washington, DC",22
5,"London, UK",21
6,New York City,19
7,Seattle,16
8,"Dublin, Ireland",15
9,São Paulo,10


In [ ]:
## workplace type

workplace_counts = (
    df["Workplace Type"]
    .value_counts()
    .reset_index()
)

workplace_counts.columns = ["Workplace Type", "Job Count"]

workplace_counts

In [71]:
## Salary analysis

salary_df = df.dropna(
    subset=["Salary Min", "Salary Max"]
).copy()

salary_df.shape

(670, 11)

In [72]:
salary_df[
    ["Salary Min", "Salary Max", "Salary Midpoint"]
].describe()

,Salary Min,Salary Max,Salary Midpoint
count,670.000000,670.000000,670.000000
mean,263270.269493,364486.962254,313878.615873
std,73362.852998,97706.172621,79190.737062
min,0.000000,70.000000,63.500000
25%,216500.000000,285000.000000,249625.000000
50%,257000.000000,370000.000000,307500.000000
75%,295000.000000,445000.000000,360000.000000
max,675000.000000,850000.000000,712500.000000


In [73]:
## average salary by department

salary_by_department = (
    salary_df
    .groupby("Department")["Salary Midpoint"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

salary_by_department

,count,mean,median
Department,,,
Safety Systems,13,403307.692308,440000.00
Research,70,366621.428571,381500.00
Data Science,19,361815.789474,365000.00
Core Product & Platform,8,356125.000000,355500.00
Partnerships,10,351800.000000,351500.00
Security,30,349473.333333,355500.00
Applied AI,105,348300.000000,355500.00
Product Management,9,343500.000000,339000.00
Technical Program Management,14,336892.857143,351000.00


In [ ]:
## top 10 highest paying job listing

top_salary_jobs = (
    salary_df[
        ["Job Title", "Department", "Location", "Salary Min",
         "Salary Max", "Salary Midpoint"]
    ]
    .sort_values("Salary Midpoint", ascending=False)
    .head(10)
)

top_salary_jobs

In [74]:
## job posting activity

df["Published Year"] = df["Published Date"].dt.year
year_counts = (
    df["Published Year"]
    .value_counts()
    .sort_index()
)

year_counts

Published Year
2023      3
2024      9
2025     69
2026    726
Name: count, dtype: int64

In [75]:
monthly_counts = (
    df.set_index("Published Date")
    .resample("ME")
    .size()
)

monthly_counts.tail(15)

Published Date
2025-07-31 00:00:00+00:00      7
2025-08-31 00:00:00+00:00      3
2025-09-30 00:00:00+00:00      9
2025-10-31 00:00:00+00:00     11
2025-11-30 00:00:00+00:00      6
2025-12-31 00:00:00+00:00      7
2026-01-31 00:00:00+00:00     21
2026-02-28 00:00:00+00:00     18
2026-03-31 00:00:00+00:00     20
2026-04-30 00:00:00+00:00     33
2026-05-31 00:00:00+00:00     49
2026-06-30 00:00:00+00:00     81
2026-07-31 00:00:00+00:00    116
2026-08-31 00:00:00+00:00    239
2026-09-30 00:00:00+00:00    149
Freq: ME, dtype: int64

In [76]:
df.shape

(807, 12)

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 807 entries, 0 to 806
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   Job Title        807 non-null    object             
 1   Department       807 non-null    object             
 2   Team             807 non-null    object             
 3   Employment Type  807 non-null    object             
 4   Location         807 non-null    object             
 5   Workplace Type   807 non-null    object             
 6   Published Date   807 non-null    datetime64[ns, UTC]
 7   Salary Min       670 non-null    float64            
 8   Salary Max       670 non-null    float64            
 9   Job URL          807 non-null    object             
 10  Salary Midpoint  670 non-null    float64            
 11  Published Year   807 non-null    int32              
dtypes: datetime64[ns, UTC](1), float64(3), int32(1), object(7)
memory usage: 72.6+